In [1]:
# Kaggle-optimized install
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 31.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 87.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.9/181.9 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 39.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requi

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Critical for Llama-3 DPO — prevents tensor type errors
tokenizer.padding_side = "right"
tokenizer.pad_token = tokenizer.eos_token

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.4: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/220 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/345 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 64,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",  # Saves VRAM
    random_state = 3407,
)

Unsloth 2026.3.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [4]:
from datasets import load_dataset
import os

data_path = "orca_rlhf.jsonl"

if os.path.exists(data_path):
    dataset = load_dataset("json", data_files=data_path, split="train")
    print(f"✅ Loaded local: {data_path}")
else:
    dataset = load_dataset("Intel/orca_dpo_pairs", split="train")
    print("⚠️ Using HuggingFace fallback")

dataset = dataset.select(range(1000))

def format_dpo(sample):
    return {
        "prompt": (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"You are a helpful assistant.<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{sample['question']}<|eot_id|>"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
        ),
        "chosen":   f"{sample['chosen']}<|eot_id|>",
        "rejected": f"{sample['rejected']}<|eot_id|>",
    }

dataset = dataset.map(format_dpo, remove_columns=dataset.column_names)
print(f"✅ Dataset formatted. Sample: {dataset[0]['prompt'][:100]}")

README.md:   0%|          | 0.00/196 [00:00<?, ?B/s]

orca_rlhf.jsonl:   0%|          | 0.00/36.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/12859 [00:00<?, ? examples/s]

⚠️ Using HuggingFace fallback


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

✅ Dataset formatted. Sample: <|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful assistant.<|eot_id|><


In [8]:
import os
# Fix memory fragmentation (This is a lifesaver for T4 GPUs)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from trl import DPOConfig, DPOTrainer
from unsloth import PatchDPOTrainer
import wandb
import torch

# 1. Patch for 2x speed
PatchDPOTrainer()

# 2. WandB
wandb.init(project="llama3-dpo-alignment", name="orca-dpo-vram-victory")

# 3. Ultra-Lean Config for T4
training_args = DPOConfig(
    output_dir = "outputs",
    beta = 0.1,
    per_device_train_batch_size = 1,   
    gradient_accumulation_steps = 8,  
    learning_rate = 5e-6,
    num_train_epochs = 1,
    optim = "paged_adamw_8bit",
    report_to = "wandb",
    logging_steps = 1,
    # ⬇️ Trimmed length to save 2GB of VRAM
    max_length = 768,                 
    max_prompt_length = 384,
    gradient_checkpointing = True,     
    # ⬇️ Disable logging to console to save minor VRAM
    logging_first_step = True,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
)

# 4. Initialize Trainer
trainer = DPOTrainer(
    model = model,
    ref_model = None, 
    args = training_args,
    train_dataset = dataset,
    tokenizer = tokenizer,
)

# 🚀 This time it fits!
trainer_stats = trainer.train()


train/epoch,▁
train/global_step,▁
train/grad_norm,▁
train/learning_rate,▁
train/logits/chosen,▁
train/logits/rejected,▁
train/logps/chosen,▁
train/logps/rejected,▁
train/loss,▁
train/rewards/accuracies,▁
+3,...


Tokenizing train dataset (num_proc=8):   0%|          | 0/1000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 125
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 167,772,160 of 8,198,033,408 (2.05% trained)


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected,eval_logits / chosen,eval_logits / rejected,nll_loss
1,0.693882,-0.002346,-0.000885,0.250000,-0.001460,-161.120285,-190.805923,0.133447,0.012228,0,0,0
2,0.691699,-0.000774,-0.003687,0.625000,0.002912,-156.879822,-195.186234,-0.085723,0.021809,No Log,No Log,No Log
3,0.690818,0.004485,-0.000199,0.750000,0.004684,-265.061768,-214.174194,-0.211455,-0.218241,No Log,No Log,No Log
4,0.690901,0.002573,-0.001942,0.875000,0.004515,-114.733437,-208.854736,0.079551,-0.251433,No Log,No Log,No Log
5,0.688177,0.001929,-0.008048,1.000000,0.009977,-187.067535,-222.862274,-0.054866,-0.165493,No Log,No Log,No Log
6,0.688251,0.001546,-0.008289,0.875000,0.009835,-248.805664,-223.749207,0.174407,-0.056542,No Log,No Log,No Log
7,0.682304,-0.000728,-0.022706,0.750000,0.021977,-153.137970,-246.741791,-0.313183,-0.180895,No Log,No Log,No Log
8,0.676217,0.008745,-0.025516,1.000000,0.034261,-173.975220,-196.581009,-0.025172,-0.164375,No Log,No Log,No Log
9,0.661271,0.005905,-0.059247,1.000000,0.065152,-81.572800,-224.553162,-0.060939,-0.269558,No Log,No Log,No Log
10,0.648061,-0.007233,-0.099694,1.000000,0.092461,-315.359589,-210.845642,-0.239295,-0.057900,No Log,No Log,No Log


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇█
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,█▇▇▇▇▇▆▆▅▃▃▃▂▂▁▂▃▄▆▂▁▃▁▄▂▂▁▁▂▂▁▂▂▅▁▁▁▂▂▃
train/learning_rate,▅▆████▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▁
train/logits/chosen,▃▄▃▅▄▄▃▅▃▅▁▅▃▃▁█▅▅▄▁▁▁▅▄▆▄▃▃▄▂▃▅▂▄▂▄▄▄▄▄
train/logits/rejected,▂▅▄▆▃▁▆▄▇▆▅▄▆▂▄▆▇▄▄▇▁▃▃▅▆▅▄▅▆█▃▅▃▃▆▅▇▄▅▃
train/logps/chosen,▆▇▄▂▄▃▅▇▄▄▇▃▄▇▆▇▆▆▆▅▆▆▃▂▁▄▇▆▆▅▄▇▅▅▁▄▂▂█▄
train/logps/rejected,█▇▇▇▇█▇▇▇▄▅▅▆▅▆▅▄▇▇▁▆▃▂▆▆▅▅▅▄▅▂▃▆▇▅▇▅▄▃▄
train/loss,█████▇▇▆▄▄▃▃▃▂▂▃▃▂▂▁▂▂▁▂▁▂▁▂▁▂▁▃▁▂▂▁▂▁▁▁
train/rewards/accuracies,▁█████████████████▇███████████▇██▇██████
+3,...


In [11]:
import logging
import warnings
from unsloth import FastLanguageModel

# 1. Silence the noisy library warnings once and for all
logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# 2. Switch to Fast Inference
FastLanguageModel.for_inference(model) 

# 3. Clean Test Prompt
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Explain why it is important to be kind to others."},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. Generate without the "Logging Traceback"
outputs = model.generate(
    **inputs, 
    max_new_tokens = 256,
    use_cache = True,
)

# 5. Show only the Pure Result
print("\n--- MODEL RESPONSE ---\n")
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
# We split on "assistant" to show only the new part if it repeats the prompt
if "assistant" in response:
    print(response.split("assistant")[-1].strip())
else:
    print(response)


--- MODEL RESPONSE ---

Being kind to others is crucial for numerous reasons. Here are some key benefits:

1. Strengthens Relationships: Kindness fosters strong, meaningful connections with others. When we show empathy and compassion, we build trust, and people are more likely to reciprocate with kindness.

2. Boosts Mental Health: Practicing kindness can improve our mental well-being by reducing stress, anxiety, and depression. It also releases endorphins, which are natural mood-boosters.

3. Encourages Empathy: When we're kind to others, we're more likely to put ourselves in their shoes and understand their struggles. This increases our capacity for empathy, allowing us to be more compassionate and understanding.

4. Fosters a Positive Environment: Kindness has a ripple effect, creating a positive atmosphere where others feel comfortable being themselves. This can lead to a more harmonious and supportive community.

5. Inspires Kindness in Others: When we show kindness, we set an ex

In [13]:
# 1. Save ONLY the LoRA adapters (Tiny: ~200MB)
# This is much better for your project folder!
model.save_pretrained("llama3_dpo_adapter")
tokenizer.save_pretrained("llama3_dpo_adapter")

# 2. Zip the tiny folder
import shutil
shutil.make_archive("llama3_dpo_adapter", 'zip', "llama3_dpo_adapter")
print("✅ Done! You can now download 'llama3_dpo_adapter.zip' instantly.")

✅ Done! You can now download 'llama3_dpo_adapter.zip' instantly.


In [14]:
import os
import shutil

# Delete the massive 16GB merged folder and zip to free up space
if os.path.exists("llama_3_dpo_aligned_lora"):
    shutil.rmtree("llama_3_dpo_aligned_lora")
    
if os.path.exists("llama3_dpo_project.zip"):
    os.remove("llama3_dpo_project.zip")

print("✅ Massive files deleted. Your disk is now clean!")

✅ Massive files deleted. Your disk is now clean!
